## Data Loading

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from collections import Counter
from sklearn import preprocessing
import numpy as np
import random

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from transformers import BertTokenizer, BertModel
import torch
import torch.nn as nn

# load_aokvqa.py
import os
import json

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

def load_aokvqa(aokvqa_dir, split, version='v1p0'):
    assert split in ['train', 'val', 'test', 'test_w_ans']
    dataset = json.load(open(
        os.path.join(aokvqa_dir, f"aokvqa_{version}_{split}.json")
    ))
    return dataset

def get_coco_path(split, image_id, coco_dir):
    return os.path.join(coco_dir, f"{split}2017", f"{image_id:012}.jpg")

AOKVQA_DIR="datasets/aokvqa/"
COCO_DIR="datasets/coco2017/"
aokvqa_dir = f"./aokvqa/{AOKVQA_DIR}"
coco_dir = f"./aokvqa/{COCO_DIR}"

Using device: cpu


In [3]:
def subsample(dataset, n_samples=None, frac=None, random_seed=42):
    random.seed(random_seed)
    if n_samples:
        return random.sample(dataset, n_samples)
    elif frac:
        sample_size = int(len(dataset) * frac)
        return random.sample(dataset, sample_size)
    return dataset

In [5]:
USE_SUBSET_DATA = True 
train_dataset = load_aokvqa(aokvqa_dir, 'train')  
val_dataset = load_aokvqa(aokvqa_dir, 'val')
test_dataset = load_aokvqa(aokvqa_dir, 'test')
print(f"Full Train aokvqa: {len(train_dataset)}")
print(f"Full Val aokvqa: {len(val_dataset)}")
print(f"Full Test aokvqa: {len(test_dataset)}")

if USE_SUBSET_DATA:
    train_dataset = subsample(train_dataset, frac=0.2)
    val_dataset = subsample(val_dataset, frac=0.2)
    test_dataset = subsample(test_dataset, frac=0.2)
    print(f"Used Train aokvqa: {len(train_dataset)}")
    print(f"Used Val aokvqa: {len(val_dataset)}")
    print(f"Used Test aokvqa: {len(test_dataset)}")

Full Train aokvqa: 17056
Full Val aokvqa: 1145
Full Test aokvqa: 6702
Used Train aokvqa: 3411
Used Val aokvqa: 229
Used Test aokvqa: 1340


## Data Preparation

Fields Considered:

- Question
- Choices
- Correct answer
- Correct Choice Indice
- Rationale
- Direct answer

In [6]:
qa_data = []
for sample in val_dataset:
    question_id = sample["question_id"]
    image_id = sample["image_id"]
    question = sample["question"]
    choices = sample["choices"]
    correct_choice_idx = sample["correct_choice_idx"]
    rationales = sample.get("rationales", [])
    combined_rationale = " ".join(rationales)
    direct_answers = sample.get("direct_answers", [])
    combined_direct_answer = " ".join(direct_answers)
    correct_ans = choices[correct_choice_idx]
    qa_data.append({
        "question_id": question_id,
        "image_id": image_id,
        "question": question,
        "choices": choices,
        "correct_answer": correct_ans,
        "correct_choice_idx": correct_choice_idx,
        "rationale": combined_rationale,
        "direct_answer": combined_direct_answer
    })
qa_df = pd.DataFrame(qa_data)

In [7]:
qa_df.head()


,question_id,image_id,question,choices,correct_answer,correct_choice_idx,rationale,direct_answer
0,Bp5VwMuky5fXJPzp54X6XE,466986,What industry is this man likely working in?,"[travel, construction, medical, restaurant]",restaurant,3,There is a microwave behind the man. there are...,cooking chef food service food restaurant food...
1,4cdYbJFFeKXJr6LbKgcxTa,176799,In what type of environment are they most like...,"[beach, city, rural, suburban]",city,1,They are riding in an urban environment given ...,urban playing skate park urban warm tropical u...
2,RrQ48CMWPTgqsvg2K4P7eR,569825,When actively playing what physical position w...,"[upside down, crouch, lying down, stand]",crouch,1,The person will crouch. The player is the catc...,crouch crouched crouched catcher catcher catch...
3,P6oCrLWLEpt8yyJ6x9AbAd,502336,Which city is this sign in which contains the ...,"[cologne germany, london, rome, vienna]",cologne germany,0,The transit stop is contained in germany. The ...,cologne heumarkt canton china cologne germany ...
4,M46zwhVtTDuYcfKTvw5Bsk,292225,What would this fence help to contain?,"[balls, animals, rocks, weeds]",balls,0,The fence is up so nothing will go flying outs...,balls balls tennis court balls tennis balls ba...


## Multimodal

### BLIP-2

In [10]:
from transformers import Blip2Processor, Blip2ForConditionalGeneration
from PIL import Image
import torch

# Load BLIP-2 model and processor
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")
model = Blip2ForConditionalGeneration.from_pretrained("Salesforce/blip2-opt-2.7b").to(device)


preprocessor_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/882 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.56M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/122k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/10.0G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

In [35]:
import torch
from transformers import AutoProcessor, Blip2ForConditionalGeneration

processor = AutoProcessor.from_pretrained("Salesforce/blip2-opt-2.7b")
model = Blip2ForConditionalGeneration.from_pretrained("Salesforce/blip2-opt-2.7b")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [36]:
def generate_blip2_answer(image_path, question):
    image = Image.open(image_path).convert("RGB")
    prompt = f"Question:{question} Answer:"
    inputs = processor(images=image, text=prompt, return_tensors="pt").to(device)
    print(inputs)

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=5)

    answer = processor.batch_decode(output, skip_special_tokens=True)[0]
    
    return answer.strip()

In [37]:
blip2_results = []
count = 1
for sample in val_dataset[:1]:
    print(count)
    count += 1
    image_path = get_coco_path("val", sample["image_id"], coco_dir)
    question = sample["question"]

    generated_answer = generate_blip2_answer(image_path, question)

    blip2_results.append({
        "question": question,
        "generated_answer": generated_answer,
        "correct_answer": sample["choices"][sample["correct_choice_idx"]],
        "image_id": sample["image_id"]
    })

# Convert results to DataFrame
blip2_df = pd.DataFrame(blip2_results)


1
{'input_ids': tensor([[50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265,
         50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265,
         50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265,
         50265, 50265,     2, 45641,    35,  2264,   539,    16,    42,   313,
           533,   447,    11,   116, 31652,    35]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]), 'pixel_values': tensor([[[[1.1712, 1.1858, 0.9230,  ..., 0.8938, 0.5289, 0.4705],
          [1.1712, 1.1858, 0.9084,  ..., 0.9668, 0.6311, 0.4121],
          [1.1566, 1.1712, 0.8063,  ..., 0.9522, 0.6603, 0.4997],
          ...,
          [0.0763, 0.1639, 0.1201,  ..., 1.9303, 1.9303, 1.9303],
          [0.2223, 0.1785, 0.1931,  ..., 1.9303, 1.9303, 1.9303],
          [0.2807, 0.2369, 0.2369,  ..., 1.9303, 1.9303, 1.9303]],

        

In [40]:
blip2_df['generated_answer']

0    Question:What industry is this man likely work...
Name: generated_answer, dtype: object

In [41]:
from sklearn.metrics import accuracy_score

blip2_df["is_correct"] = blip2_df["generated_answer"].str.lower() == blip2_df["correct_answer"].str.lower()
accuracy = blip2_df["is_correct"].mean()
print(f"BLIP-2 Accuracy: {accuracy:.2%}")

BLIP-2 Accuracy: 0.00%


In [ ]:
import json

# Path to save results
output_json_path = "blip2_vqa_results.json"

# Process dataset and store results
blip2_results = []
for sample in val_dataset:
    image_path = get_coco_path("val", sample["image_id"], coco_dir)
    question = sample["question"]

    generated_answer = generate_blip2_answer(image_path, question)

    result = {
        "question_id": sample["question_id"],
        "image_id": sample["image_id"],
        "question": question,
        "generated_answer": generated_answer,
        "correct_answer": sample["choices"][sample["correct_choice_idx"]],
        "choices": sample["choices"]
    }
    blip2_results.append(result)

# Save results to JSON file
with open(output_json_path, "w") as f:
    json.dump(blip2_results, f, indent=4)

print(f"BLIP-2 VQA results saved to {output_json_path}")

In [ ]:
with open(output_json_path, "r") as f:
    loaded_results = json.load(f)

# Print the first 5 entries
print(json.dumps(loaded_results[:5], indent=4))

In [ ]:
import gc
import torch

# Delete the model and processor from memory
del model
del processor

# Manually free up memory
gc.collect()

# If using GPU, clear CUDA memory
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("BLIP-2 model removed from memory.")

### CLIP

In [8]:
def get_clip_prediction(img_path, choices, question):
    # Load and preprocess the image
    image = Image.open(img_path).convert("RGB")
    image_input = preprocess(image).unsqueeze(0).to(device)
    
    # Tokenize the list of candidate text choices
    # text_input = clip.tokenize(choices).to(device)
    text_input = torch.cat([
        clip.tokenize(f"{question} Answer: {c}") for c in choices
    ]).to(device)
    # print(str(text_input))
    
    with torch.no_grad():
        # Compute image and text features
        image_features = model.encode_image(image_input)
        text_features = model.encode_text(text_input)
        
        # Normalize features to unit length (recommended for cosine similarity)
        image_features /= image_features.norm(dim=-1, keepdim=True)
        text_features /= text_features.norm(dim=-1, keepdim=True)
        
        # Compute similarity between the image and each text choice
        similarity = (image_features @ text_features.T).squeeze(0)

        confidence_scores = torch.softmax(similarity, dim=0).cpu().numpy()
    
    # Return the choice with the highest similarity score
    best_idx = similarity.argmax().item()
    best_answer = choices[best_idx]
    best_confidence = confidence_scores[best_idx]
    
    return best_answer, best_confidence

In [9]:
import torch
import clip
from PIL import Image
from tqdm import tqdm

# Set up device and load the CLIP model along with its preprocessing pipeline.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model, preprocess = clip.load("ViT-B/32", device=device)
model.eval()

# Process each sample in qa_data and compute predictions
predictions = []
condifence_scores = []
for sample in tqdm(qa_data, desc="Processing Images"):
    image_id = sample['image_id']
    img_path = get_coco_path('val', image_id, coco_dir)
    choices  = sample['choices'] 
    prediction, confidence = get_clip_prediction(img_path, choices, sample['question'])
    predictions.append(prediction)
    condifence_scores.append(confidence)

qa_df['clip_baseline_prediction'] = predictions
qa_df['clip_baseline_confidence'] = condifence_scores


ModuleNotFoundError: No module named 'clip'

In [23]:
correctness = (qa_df['clip_baseline_prediction'] == qa_df['correct_answer'])
accuracy = correctness.mean()
calibration_error = (qa_df['clip_baseline_confidence'] - correctness).abs().mean()
print(f"CLIP Baseline Accuracy: {accuracy:.2%}")
print(f"CLIP Baseline Calibration Error: {calibration_error:.2%}")

CLIP Baseline Accuracy: 55.11%
CLIP Baseline Calibration Error: 52.41%


# VisualBERT
## Multiple Choice

In [ ]:
def extract_visual_features(img_path, max_regions=36):
    """Extract region features using Faster R-CNN."""
    img = Image.open(img_path).convert("RGB")
    img_array = np.array(img)
    outputs = predictor(img_array)
    
    # Get top-k regions by objectness score
    boxes = outputs["instances"].pred_boxes.tensor.cpu().numpy()
    features = outputs["instances"].pred_features.cpu().numpy()  # Shape: [N, 2048]
    scores = outputs["instances"].scores.cpu().numpy()
    
    # Select top-k regions
    top_k = min(max_regions, len(scores))
    indices = np.argsort(scores)[-top_k:]
    features = features[indices]
    boxes = boxes[indices]
    
    # Normalize boxes (xyxy to xywh) and normalize coordinates
    img_width, img_height = img.size
    boxes[:, 2] -= boxes[:, 0]  # x2 -> w
    boxes[:, 3] -= boxes[:, 1]  # y2 -> h
    boxes[:, [0, 2]] /= img_width   # Normalize x, w
    boxes[:, [1, 3]] /= img_height  # Normalize y, h
    
    # Combine boxes and features (VisualBERT expects [num_regions, 2048 + 4])
    visual_embeds = np.concatenate([features, boxes], axis=1)
    visual_embeds = torch.from_numpy(visual_embeds).float().unsqueeze(0).to(device)  # [1, 36, 2052]
    
    return visual_embeds

def visualbert_multiple_choice(img_path, question, choices):
    # Get image features (REPLACE THIS WITH REAL OBJECT DETECTOR OUTPUT)
    visual_embeds = extract_visual_features(img_path)
    visual_attention_mask = torch.ones(visual_embeds.shape[:2]).to(device)
    visual_token_type_ids = torch.ones(visual_embeds.shape[:2]).to(device)
    
    # Prepare inputs for each choice
    input_ids, attention_masks, token_type_ids = [], [], []
    
    for choice in choices:
        text = f"{question} [SEP] {choice}"
        inputs = tokenizer(
            text,
            return_tensors="pt",
            padding="max_length",
            max_length=128,
            truncation=True
        )
        input_ids.append(inputs["input_ids"])
        attention_masks.append(inputs["attention_mask"])
        token_type_ids.append(inputs["token_type_ids"])
    
    # Stack inputs
    input_ids = torch.cat(input_ids, dim=0).to(device)  # Shape: [num_choices, max_length]
    attention_mask = torch.cat(attention_masks, dim=0).to(device)
    token_type_ids = torch.cat(token_type_ids, dim=0).to(device)
    
    # Run VisualBERT
    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            visual_embeds=visual_embeds,
            visual_attention_mask=visual_attention_mask,
            visual_token_type_ids=visual_token_type_ids,
        )
        logits = outputs.logits
        choice_confidence_scores = torch.softmax(logits, dim=1).squeeze(0).cpu().numpy()
    
    best_idx = logits.argmax().item()

    # direct_answer:
    # Prepare inputs for direct-answering (no choices involved)
    text = f"{question}"
    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding="max_length",
        max_length=128,
        truncation=True
    )
    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)
    token_type_ids = inputs["token_type_ids"].to(device)

    # Run VisualBERT for direct-answering
    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            visual_embeds=visual_embeds,
            visual_attention_mask=visual_attention_mask,
            visual_token_type_ids=visual_token_type_ids,
        )
        logits = outputs.logits
        direct_answer = tokenizer.decode(logits.argmax(dim=-1), skip_special_tokens=True)  # Decode answer
        direct_confidence_score = torch.softmax(logits, dim=1).squeeze(0).cpu().numpy()

    return choices[best_idx], float(choice_confidence_scores[best_idx]), direct_answer, direct_confidence_score

In [26]:
import torch
from PIL import Image
from transformers import VisualBertForMultipleChoice, BertTokenizer
from torchvision import transforms

model = VisualBertForMultipleChoice.from_pretrained("uclanlp/visualbert-vqa", ignore_mismatched_sizes=True)
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")  # VisualBERT uses BERT's tokenizer
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

import torch
import numpy as np
from PIL import Image
from detectron2 import model_zoo
from detectron2.engine import DefaultPredictor
from detectron2.config import get_cfg

# Initialize Faster R-CNN detector
cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("COCO-Detection/faster_rcnn_R_101_FPN_3x.yaml"))
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5  # Threshold
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-Detection/faster_rcnn_R_101_FPN_3x.yaml")
predictor = DefaultPredictor(cfg)

Some weights of VisualBertForMultipleChoice were not initialized from the model checkpoint at uclanlp/visualbert-vqa and are newly initialized because the shapes did not match:
- cls.weight: found shape torch.Size([3129, 768]) in the checkpoint and torch.Size([1, 768]) in the model instantiated
- cls.bias: found shape torch.Size([3129]) in the checkpoint and torch.Size([1]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


RuntimeError: No CUDA GPUs are available